In [ ]:
import geopandas as gpd
import pandas as pd
import shapely
from shapely.geometry import box

from paths import paths, prepare_preprocessing

In [ ]:
prepare_preprocessing()

In [ ]:
polygons = gpd.read_parquet(paths.preprocessing.shared.lsoa)
polygons.reset_index(inplace = True, drop = True)

In [ ]:
# Load the pollution data and merge into single GDF
def load_grid(path, value_col):
    df = pd.read_csv(path, skiprows = 4)
    df.columns = df.iloc[0]
    df = df[1:]

    df.rename(columns = {value_col: 'value'}, inplace = True)
    df['value'] = pd.to_numeric(df['value'], errors = 'coerce')

    gdf = gpd.GeoDataFrame(df, geometry = gpd.points_from_xy(df['x'], df['y'], crs = 27700))
    gdf['geometry'] = gdf.geometry.apply(lambda p: box(p.x - 500, p.y - 500, p.x + 500, p.y + 500))

    return gdf[['geometry', 'value']]

nox = load_grid(paths.raw.context.nox, 'nox2023')
pm25 = load_grid(paths.raw.context.pm25, 'pm252023g')
ozone = load_grid(paths.raw.context.ozone, 'dgt12023')

grid = (
    nox.rename(columns = {'value': 'nox'})
    .merge(pm25.rename(columns = {'value': 'pm25'}), on = 'geometry')
    .merge(ozone.rename(columns = {'value': 'ozone'}), on = 'geometry')
)

In [ ]:
# Intersect LSOAs with pollution grid, calculate intersection area, and compute weighted average concentrations
joined = gpd.sjoin(polygons[['LSOA21CD', 'geometry']], grid, how = 'inner', predicate = 'intersects')

lsoa_i = joined.index.to_numpy()
grid_i = joined['index_right'].to_numpy()

lsoa_geom = polygons.geometry.take(lsoa_i).array
grid_geom = grid.geometry.take(grid_i).array

intersection_area = shapely.area(shapely.intersection(lsoa_geom, grid_geom))

overlap_df = joined[['LSOA21CD']].copy()
overlap_df['intersection_area'] = intersection_area

overlap_df['nox'] = grid['nox'].take(grid_i).to_numpy()
overlap_df['pm25'] = grid['pm25'].take(grid_i).to_numpy()
overlap_df['ozone'] = grid['ozone'].take(grid_i).to_numpy()

lsoa_area = polygons.set_index('LSOA21CD').geometry.area
overlap_df['w'] = overlap_df['intersection_area'] / overlap_df['LSOA21CD'].map(lsoa_area)

for pol in ['nox', 'pm25', 'ozone']:
    overlap_df[f'{pol}_w'] = overlap_df[pol] * overlap_df['w']

result = (
    overlap_df.groupby('LSOA21CD')[['nox_w', 'pm25_w', 'ozone_w']].sum().rename(columns = lambda c: c[:-2]).reset_index()
)

result.to_csv(paths.preprocessing.context.pollution, index = False)